# 02 — Dataset Quality Audit

> **Notebook status: documented reconstruction.** The project history records the quality-audit methods and the 183-image review queue. The original V2 audit notebook and complete per-image audit manifest were not recovered. The code below is a reusable audit implementation; it must not be mistaken for the exact run that produced the historical count of 183 flagged images.

## Purpose

The quality audit was designed to identify files needing inspection before training, not to decide whether a photograph was authentic. First-pass checks were applied to all **1,250 Dataset V2 candidates**. The audit identified **183 images for review**. Those flags were not automatic deletion decisions. Readable and useful images were retained even when composition, aspect ratio or object-detection output was imperfect.

## Checks recorded in the project history

- Image decode/readability.
- Width, height, pixel count and aspect ratio.
- Brightness and contrast summaries.
- Blur indicator.
- SHA-256 file hash.
- Decoded-pixel hash.
- Perceptual hash (pHash) for near-duplicate review.
- Car object-detection screening result.
- Quality flags and review-priority output.
- Contact sheets for manual review.

Object detection was a screening aid only. A missed detection is not proof that a car is absent or that the image is unusable. Likewise, a successful detection is not proof that the image is genuine.

## What the review count means

The historical count **183** means images were queued for additional review, not that 183 images were deleted. Quality flags were heuristic indicators. The project intentionally retained readable, usable images despite some imperfect framing, unusual ratios or detector failures.

## Historical limitations

- No complete per-image V2 audit manifest, detector checkpoint, or exact flag thresholds were recovered with the source archive.
- Real-image authenticity was not independently verified for every image.
- AI labels and generator identities were inherited from source folder/manifest labels.
- This notebook's configurable heuristics must not be presented as the exact historical settings.


In [ ]:
from pathlib import Path
import hashlib, math
import numpy as np
import pandas as pd
from PIL import Image, ImageOps, ImageStat, UnidentifiedImageError, ImageFilter

DATASET_ROOT = Path("data/dataset_v2")  # External; the 1,250 images are not in this repository.
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".gif", ".tif", ".tiff"}
HISTORICAL_CANDIDATES = 1250
HISTORICAL_REVIEW_QUEUE = 183

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def pixel_sha256(image: Image.Image) -> str:
    rgb = ImageOps.exif_transpose(image).convert("RGB")
    return hashlib.sha256(f"{rgb.width}x{rgb.height}:".encode() + rgb.tobytes()).hexdigest()

def measure_image(path: Path) -> dict:
    row = {"path": str(path), "readable": False, "error": None}
    try:
        with Image.open(path) as raw:
            raw.seek(0)
            image = ImageOps.exif_transpose(raw).convert("RGB")
            stat = ImageStat.Stat(image)
            gray = image.convert("L")
            # PIL's FIND_EDGES mean is a simple blur/edge proxy; it is not a calibrated blur metric.
            edge_mean = ImageStat.Stat(gray.filter(ImageFilter.FIND_EDGES)).mean[0]
            row.update({
                "readable": True,
                "width": image.width,
                "height": image.height,
                "pixels": image.width * image.height,
                "aspect_ratio": image.width / image.height if image.height else None,
                "brightness_mean": float(ImageStat.Stat(gray).mean[0]),
                "contrast_std": float(ImageStat.Stat(gray).stddev[0]),
                "edge_mean_proxy": float(edge_mean),
                "pixel_sha256": pixel_sha256(image),
                "file_sha256": sha256_file(path),
            })
    except Exception as exc:
        row["error"] = f"{type(exc).__name__}: {exc}"
    return row

if DATASET_ROOT.exists():
    image_paths = sorted(p for p in DATASET_ROOT.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)
    audit_df = pd.DataFrame(measure_image(p) for p in image_paths)
    display(audit_df.head())
    print("Images scanned:", len(audit_df))
else:
    audit_df = pd.DataFrame()
    print(f"Dataset folder not found: {DATASET_ROOT}")
    print(f"Historical audit: {HISTORICAL_CANDIDATES} candidates; {HISTORICAL_REVIEW_QUEUE} queued for review.")


## Perceptual hashes (optional dependency)

pHash is used to locate visually similar candidate pairs for review. A close pHash distance is a review cue, not a proof of duplication. Do not auto-delete based on pHash alone.

In [ ]:
try:
    import imagehash
    from PIL import Image
    def phash_for(path: Path):
        with Image.open(path) as im:
            return str(imagehash.phash(ImageOps.exif_transpose(im).convert("RGB")))
    if not audit_df.empty:
        audit_df["phash"] = [phash_for(Path(p)) if readable else None for p, readable in zip(audit_df.path, audit_df.readable)]
except ImportError:
    print("Install ImageHash in the notebook environment to calculate pHash. No pHash values are fabricated.")


## Quality flags: thresholds are intentionally configurable

The original V2 thresholds were not recovered, so the defaults below are `None` and will not invent cut-offs. Fill them only if the original audit configuration is available. A flag sends an image to review; it never deletes an image.

In [ ]:
RULES = {
    "min_width": None,
    "min_height": None,
    "min_aspect_ratio": None,
    "max_aspect_ratio": None,
    "min_brightness_mean": None,
    "max_brightness_mean": None,
    "min_edge_mean_proxy": None,
}

def apply_review_rules(frame: pd.DataFrame, rules: dict) -> pd.DataFrame:
    if frame.empty:
        return frame.copy()
    out = frame.copy()
    out["quality_flags"] = ""
    for column, threshold in rules.items():
        if threshold is None or column not in out.columns:
            continue
        if column.startswith("min_"):
            metric = column[4:]
            mask = out[metric].notna() & (out[metric] < threshold)
        elif column.startswith("max_"):
            metric = column[4:]
            mask = out[metric].notna() & (out[metric] > threshold)
        else:
            continue
        out.loc[mask, "quality_flags"] = out.loc[mask, "quality_flags"].map(lambda x: (x + ";" if x else "") + column)
    out.loc[~out["readable"], "quality_flags"] = out.loc[~out["readable"], "quality_flags"].map(lambda x: (x + ";" if x else "") + "unreadable")
    out["review_required"] = out["quality_flags"].ne("")
    return out

if not audit_df.empty:
    audited = apply_review_rules(audit_df, RULES)
    display(audited["review_required"].value_counts(dropna=False))
else:
    print("No current image data: historical review count remains 183, not recomputed here.")


## Car-detection screening and manual review

The original detector model and exact detector threshold are not present in the recovered source files. For that reason, this notebook does not silently substitute a different detector. If the historical audit manifest is available, import its `car_detection_status` / `car_detected` field as evidence. A detector miss should remain a flag for human review, not an automatic exclusion.

A review-priority table can combine flags, image dimensions, blur/brightness proxies, duplicate pairs and detector fields. Keep human decision columns such as `review_status`, `reviewer_note` and `retained_after_review`.

In [ ]:
if not audit_df.empty:
    # Example review queue: unreadable images first, then rows with the most configured flags.
    review_queue = apply_review_rules(audit_df, RULES).copy()
    review_queue["priority"] = review_queue["quality_flags"].fillna("").str.count(";") + review_queue["quality_flags"].ne("").astype(int)
    review_queue = review_queue.sort_values(["readable", "priority"], ascending=[True, False])
    review_queue["review_status"] = "pending_manual_review"
    output_dir = Path("documentation/generated_audit")
    output_dir.mkdir(parents=True, exist_ok=True)
    review_queue.to_csv(output_dir / "review_priority_reconstructed.csv", index=False)
    audit_df.to_csv(output_dir / "image_audit_reconstructed.csv", index=False)
    print("Wrote a new audit. This output is not the historic 183-row review manifest.")


## Contact sheets

Contact sheets support human review. They are visual indexes, not automated authenticity checks. The code below creates a small sheet from images that are available locally.

In [ ]:
from PIL import Image, ImageOps, ImageDraw
import math

def make_contact_sheet(paths, output_path: Path, columns: int = 5, thumb_size=(180, 130)):
    paths = list(paths)
    if not paths:
        print("No images available for a contact sheet.")
        return None
    rows = math.ceil(len(paths) / columns)
    label_h = 30
    sheet = Image.new("RGB", (columns * thumb_size[0], rows * (thumb_size[1] + label_h)), "white")
    draw = ImageDraw.Draw(sheet)
    for i, path in enumerate(paths):
        try:
            with Image.open(path) as raw:
                thumb = ImageOps.fit(ImageOps.exif_transpose(raw).convert("RGB"), thumb_size)
            x, y = (i % columns) * thumb_size[0], (i // columns) * (thumb_size[1] + label_h)
            sheet.paste(thumb, (x, y))
            draw.text((x + 3, y + thumb_size[1] + 5), Path(path).name[:26], fill="black")
        except Exception:
            continue
    output_path.parent.mkdir(parents=True, exist_ok=True)
    sheet.save(output_path, quality=90)
    return output_path

if DATASET_ROOT.exists():
    candidates = [p for p in DATASET_ROOT.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS][:50]
    make_contact_sheet(candidates, Path("documentation/generated_audit/contact_sheet_sample.jpg"))
else:
    print("Contact sheet not generated because source images are not included.")


## Interpretation

The historical 183-image review queue is preserved as a project result. This notebook's code is a reusable implementation of the audit concepts; because the original per-image V2 manifest, exact heuristic thresholds and car detector artefact were not recovered, a rerun cannot be expected to return the same 183 items. No image is deleted by this notebook.